# ICS40125 - Laboratorio N°07

Machine Learning: regresión (California Housing) y clasificación (dígitos).

## Regresión — California Housing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")

housing_data = fetch_california_housing(as_frame=True)
housing = housing_data['data']
housing['target'] = housing_data['target']
housing.head()

### 1-3. Exploración y visualización

In [ ]:
# estadistica descriptiva
print(housing.shape)
print(housing.isnull().sum())
housing.describe()

In [ ]:
# correlacion con el target
plt.figure(figsize=(10,8))
sns.heatmap(housing.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlacion')
plt.show()
# MedInc (ingreso medio) es la variable mas correlacionada con el valor de la vivienda.

In [ ]:
# distribucion del target
plt.figure(figsize=(8,4))
sns.histplot(housing['target'], bins=40)
plt.title('Distribucion del valor medio de vivienda')
plt.show()

### 4. Preprocesamiento

In [ ]:
X = housing.drop(columns='target')
y = housing['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# escalar (importante para KNN y regresion lineal)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("train:", X_train.shape, "| test:", X_test.shape)

### 5-6. Comparación de modelos y métricas

In [ ]:
modelos = {
    'LinearRegression': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'KNN': KNeighborsRegressor(n_neighbors=5)
}

resultados = []
predicciones = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train_s, y_train)
    y_pred = modelo.predict(X_test_s)
    predicciones[nombre] = y_pred
    resultados.append({
        'modelo': nombre,
        'mae': round(mean_absolute_error(y_test, y_pred), 4),
        'rmse': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'r2': round(r2_score(y_test, y_pred), 4)
    })

df_resultados = pd.DataFrame(resultados).sort_values('r2', ascending=False)
df_resultados

### 7. Visualización de resultados

In [ ]:
# mejor modelo: real vs predicho
mejor = df_resultados.iloc[0]['modelo']
plt.figure(figsize=(6,6))
plt.scatter(y_test, predicciones[mejor], alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('real'); plt.ylabel('predicho')
plt.title(f'Real vs Predicho ({mejor})')
plt.show()

### 8. Conclusiones (regresión)

Random Forest suele dar el mejor $R^2$, porque captura relaciones no lineales entre las variables. La regresión lineal queda por debajo pero sirve como línea base. La variable más influyente es el ingreso medio de la zona (`MedInc`).

## Clasificación — Dígitos

In [ ]:
import numpy as np
import pandas as pd
from sklearn import datasets
import matplotlib.pyplot as plt

%matplotlib inline

digits_dict = datasets.load_digits()
digits = (
    pd.DataFrame(digits_dict["data"])
    .rename(columns=lambda x: f"c{x:02d}")
    .assign(target=digits_dict["target"])
    .astype(int)
)
digits.head()

### Ejercicio 1 — Análisis exploratorio

In [ ]:
print("Dimensiones:", digits.shape)
print("Memoria (KB):", round(digits.memory_usage(deep=True).sum() / 1024, 1))
print("Tipos:", digits.dtypes.unique())
print()
print("Registros por clase:")
print(digits['target'].value_counts().sort_index())

### Ejercicio 2 — Visualización

In [ ]:
nx, ny = 5, 5
fig, axs = plt.subplots(nx, ny, figsize=(12, 12))
for i, ax in enumerate(axs.ravel()):
    ax.imshow(digits_dict["images"][i], cmap='gray_r')
    ax.text(0, 0, str(digits_dict["target"][i]), color='red', fontsize=14)
    ax.axis('off')
plt.show()

### Ejercicio 3 — Machine Learning

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns
import time

X = digits.drop(columns="target").values
y = digits["target"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("train:", len(X_train), "| test:", len(X_test))

In [ ]:
modelos = {
    'LogisticRegression': LogisticRegression(max_iter=5000),
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42)
}

for nombre, modelo in modelos.items():
    t0 = time.time()
    modelo.fit(X_train, y_train)
    t = time.time() - t0
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{nombre}: accuracy={acc:.4f} | tiempo={t:.3f}s")

In [ ]:
# matriz de confusion del mejor modelo (KNN suele destacar en digits)
modelo = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
y_pred = modelo.predict(X_test)

plt.figure(figsize=(8,6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('predicho'); plt.ylabel('real')
plt.title('Matriz de confusion - KNN')
plt.show()

print(classification_report(y_test, y_pred))

**Respuestas Ejercicio 3:** KNN suele dar la mejor accuracy en este dataset y se ajusta muy rápido (no entrena, solo guarda los datos). RandomForest también rinde muy bien pero tarda más. Elegimos KNN por su buen balance precisión/velocidad.

### Ejercicio 4 — Curva ROC (una clase vs resto)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# binarizamos las 10 clases
clases = np.arange(10)
y_test_bin = label_binarize(y_test, classes=clases)

modelo = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
y_score = modelo.predict_proba(X_test)

plt.figure(figsize=(8,6))
for i in clases:
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    plt.plot(fpr, tpr, label=f'digito {i} (AUC={auc(fpr, tpr):.2f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('Curva ROC por clase (one-vs-rest)')
plt.legend(fontsize=8)
plt.show()
# Todas las clases tienen AUC cercano a 1: el modelo separa muy bien los digitos.

### Ejercicio 5 — Aciertos vs errores

In [ ]:
def mostrar_resultados(digits, model, nx=5, ny=5, label="correctos"):
    X = digits.drop(columns="target").values
    y = digits["target"].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if label == "correctos":
        mask = (y_pred == y_test); color = "green"
    elif label == "incorrectos":
        mask = (y_pred != y_test); color = "red"
    else:
        raise ValueError("Valor incorrecto")

    X_aux, y_true, y_hat = X_test[mask], y_test[mask], y_pred[mask]
    n = min(nx * ny, len(X_aux))
    idx = np.random.choice(len(X_aux), n, replace=False)
    fig, ax = plt.subplots(nx, ny, figsize=(12, 12))
    for i, index in enumerate(idx):
        data = X_aux[index].reshape(8, 8)
        r, c = i // ny, i % ny
        ax[r, c].imshow(data, cmap='gray_r')
        ax[r, c].text(0, 0, str(y_hat[index]), color=color, fontsize=10)
        ax[r, c].text(7, 0, str(y_true[index]), color='blue', fontsize=10)
        ax[r, c].axis('off')
    plt.show()

mostrar_resultados(digits, KNeighborsClassifier(n_neighbors=3), label="correctos")

In [ ]:
mostrar_resultados(digits, KNeighborsClassifier(n_neighbors=3), label="incorrectos")
# Los errores ocurren en digitos ambiguos o mal escritos (ej. 1 vs 7, 3 vs 8),
# donde las imagenes de 8x8 pierden detalle.

### Ejercicio 6 — Conclusiones

El dataset está balanceado (~180 imágenes por dígito) y no tiene nulos. KNN y Random Forest logran accuracy sobre 0.97. Los pocos errores se dan en dígitos visualmente parecidos. Como trabajo futuro se podría probar reducción de dimensionalidad (PCA) o redes neuronales para exprimir aún más el rendimiento.